# Real-Time Payments Fraud Detection - Interactive Walkthrough

## Overview
This notebook demonstrates **sub-500ms fraud detection** for irrevocable real-time payments (FedNow/RTP) with complete audit trail capture.

### What You'll Learn:
- Ultra-fast fraud routing for irrevocable payments
- Latency compliance monitoring (<500ms requirement)
- Irrevocability risk tracking and documentation
- Real-time payment regulatory compliance (OCC/Fed guidance)

### Regulatory Context:
- **Regulation**: OCC/Fed RTP Guidance (FedNow/RTP)
- **Regulator**: OCC/Federal Reserve
- **Requirements**: Sub-500ms decisions, irrevocability tracking, audit trails

### Critical Constraints:
- ⏱ **<500ms decision latency** (hard requirement)
- [SECURED] **Irrevocable once approved** (cannot be recalled)
- **Results:** **Complete audit trail** for all routing decisions

## Step 1: Setup and Imports

In [ ]:
import sys
import os
import uuid
import random
import time
import hashlib
from datetime import datetime, timedelta
from typing import Dict, Any

# Add shared module to path
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'shared'))

try:
    import backend
    from backend import briefcase_ai, DecisionSnapshot, Input, Output, SqliteBackend
    print("[SUCCESS] Successfully imported Briefcase AI SDK")
except ImportError as e:
    print(f"[FAILED] Error importing required modules: {e}")

## Step 2: Initialize Briefcase AI

In [ ]:
# Initialize Briefcase AI SDK
try:
    briefcase_ai.init_with_config(2)
    print("[SUCCESS] Briefcase AI SDK initialized")
except Exception as e:
    print(f"[FAILED] Failed to initialize SDK: {e}")

# Get configured backend
db_backend = backend.get_backend()
print("[SUCCESS] SQLite backend configured for RTP audit trails")

print(f"\n[URGENT] Real-Time Payment Constraints:")
print(f"  • Maximum decision latency: 500ms")
print(f"  • Payment irrevocable once approved")
print(f"  • Complete fraud routing audit required")

## Step 3: Simulate Real-Time Payment Data

Create a FedNow payment that requires ultra-fast fraud routing.

In [ ]:
# Generate a real-time payment for fraud routing
payment_id = str(uuid.uuid4())
sender_account_hash = hashlib.sha256("sender_account_12345".encode()).hexdigest()[:16]
receiver_account_hash = hashlib.sha256("receiver_account_67890".encode()).hexdigest()[:16]

payment_data = {
    "payment_id": payment_id,
    "payment_rail": "fednow",  # FedNow Service
    "sender_account_hash": sender_account_hash,
    "receiver_account_hash": receiver_account_hash,
    "payment_amount": 15000.0,  # High-value payment
    "payment_timestamp": datetime.utcnow().isoformat(),
    "sender_velocity_score": 0.25,  # Low sender risk
    "receiver_risk_score": 0.65,   # Moderate receiver risk
    "model_version": "rtp-fraud-v5.1.2",
    "routing_config_version": "rtp-routing-v2.0.4"
}

print("**Cost:** Real-Time Payment for Fraud Routing:")
for key, value in payment_data.items():
    if "hash" in key:
        print(f"  • {key}: {str(value)[:12]}...")
    else:
        print(f"  • {key}: {value}")

print(f"\n[WARNING] Risk Assessment:")
print(f"  • High value: ${payment_data['payment_amount']:,.2f}")
print(f"  • Sender velocity: {payment_data['sender_velocity_score']} (low risk)")
print(f"  • Receiver risk: {payment_data['receiver_risk_score']} (moderate risk)")
print(f"  • Payment rail: {payment_data['payment_rail'].upper()} (irrevocable)")

## Step 4: Ultra-Fast Fraud Routing Model

Simulate the AI model that must make routing decisions in <500ms.

In [ ]:
def simulate_rtp_fraud_routing(payment_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Simulates ultra-fast RTP fraud routing model.
    Must complete in <500ms for regulatory compliance.
    """
    start_time = time.time()
    
    amount = payment_data["payment_amount"]
    sender_velocity = payment_data["sender_velocity_score"]
    receiver_risk = payment_data["receiver_risk_score"]
    
    # Ultra-fast risk calculation (optimized for speed)
    risk_score = 0.0
    
    # Amount-based risk (simplified)
    if amount > 10000:
        risk_score += 0.3
    elif amount > 5000:
        risk_score += 0.1
    
    # Sender velocity risk
    if sender_velocity > 0.7:
        risk_score += 0.4
    elif sender_velocity > 0.4:
        risk_score += 0.2
    
    # Receiver risk
    risk_score += receiver_risk * 0.5
    
    # Add minimal randomness (must be fast)
    risk_score += random.uniform(-0.05, 0.05)
    risk_score = max(0.0, min(1.0, risk_score))
    
    # Fast decision thresholds
    confidence = 0.85  # High confidence for irrevocable decisions
    
    if risk_score >= 0.6:
        routing_decision = "reject"  # Too risky for irrevocable payment
    else:
        routing_decision = "approve"  # Safe to approve (irrevocable)
    
    # Calculate actual latency
    end_time = time.time()
    latency_ms = int((end_time - start_time) * 1000)
    
    return {
        "routing_decision": routing_decision,
        "confidence": round(confidence, 3),
        "risk_score": round(risk_score, 3),
        "decision_latency_ms": latency_ms,
        "latency_compliant": latency_ms <= 500,
        "irrevocable": routing_decision == "approve",
        "model_version": "rtp-fraud-v5.1.2"
    }

# Run RTP fraud routing (time-critical)
print("[URGENT] Running fraud routing model (must complete <500ms)...")
routing_result = simulate_rtp_fraud_routing(payment_data)

print(f"\n**Results:** Routing Results:")
print(f"  • Routing Decision: {routing_result['routing_decision'].upper()}")
print(f"  • Confidence: {routing_result['confidence']}")
print(f"  • Risk Score: {routing_result['risk_score']}")
print(f"  • Decision Latency: {routing_result['decision_latency_ms']}ms")

# Latency compliance check
if routing_result['latency_compliant']:
    print(f"  • [SUCCESS] Latency Compliant: {routing_result['decision_latency_ms']}ms ≤ 500ms")
else:
    print(f"  • [FAILED] Latency Violation: {routing_result['decision_latency_ms']}ms > 500ms")

# Irrevocability warning
if routing_result['irrevocable']:
    print(f"  • [SECURED] PAYMENT APPROVED - Now IRREVOCABLE (cannot be recalled)")
    print(f"  • [WARNING] No ability to reverse this decision after approval")
else:
    print(f"  • 🚫 PAYMENT REJECTED - No irrevocability risk")

## Step 5: Create RTP Compliance Audit Trail

Capture the routing decision with RTP-specific regulatory metadata.

In [ ]:
# Prepare RTP regulatory metadata
regulatory_metadata = {
    "regulation": "OCC/Fed RTP Guidance",
    "payment_rail": "fednow",
    "irrevocable": routing_result["irrevocable"],
    "latency_requirement_met": routing_result["latency_compliant"],
    "actual_latency_ms": routing_result["decision_latency_ms"],
    "max_decision_latency_ms": 500,
    "decision_timestamp": datetime.utcnow().isoformat(),
    "audit_trail_required": True,
    "examiner_ready": True
}

print("**Details:** RTP Compliance Metadata:")
for key, value in regulatory_metadata.items():
    print(f"  • {key}: {value}")

# Critical compliance warnings
if routing_result["irrevocable"]:
    print(f"\n[SECURED] IRREVOCABILITY NOTICE:")
    print(f"  • Payment cannot be recalled once processed")
    print(f"  • Fraud routing decision is FINAL")
    print(f"  • Complete audit trail captured for compliance")

# Create decision snapshot for RTP routing
try:
    decision_snapshot = backend.create_decision_snapshot(
        function_name="rtp_fraud_routing",
        inputs=payment_data,
        outputs=routing_result,
        metadata=regulatory_metadata,
        input_types={
            "payment_amount": "float",
            "sender_velocity_score": "float",
            "receiver_risk_score": "float"
        },
        output_types={
            "confidence": "float",
            "risk_score": "float",
            "decision_latency_ms": "int"
        }
    )
    print("\n[SUCCESS] RTP routing decision snapshot created")
    
except Exception as e:
    print(f"\n[FAILED] Error creating decision snapshot: {e}")

## Step 6: Store in Immutable Audit Trail

In [ ]:
# Store RTP routing decision in backend
try:
    stored_decision_id = db_backend.save_decision(decision_snapshot)
    print(f"[SUCCESS] RTP routing decision stored in audit trail")
    print(f"[SECURED] Decision ID: {stored_decision_id}")
    
    if routing_result["routing_decision"] == "approve":
        print(f"\n[SUCCESS] PAYMENT PROCESSING APPROVED")
        print(f"  • FedNow payment will be irrevocable once sent")
        print(f"  • Complete audit trail preserved")
        print(f"  • Decision latency: {routing_result['decision_latency_ms']}ms")
    else:
        print(f"\n🚫 PAYMENT REJECTED")
        print(f"  • Risk score too high for irrevocable payment")
        print(f"  • Customer will need alternative payment method")
    
except Exception as e:
    print(f"[FAILED] Error storing decision: {e}")

## Step 7: Retrieve and Display Audit Trail

In [ ]:
print("**Analysis:** RTP AUDIT TRAIL DEMONSTRATION")
print("=" * 70)

# Load decision back from backend
try:
    retrieved_decision = db_backend.load_decision(stored_decision_id)
    if retrieved_decision:
        backend.print_audit_summary(retrieved_decision)
    else:
        print("[FAILED] Failed to retrieve decision from backend")
        
except Exception as e:
    print(f"[FAILED] Error retrieving decision: {e}")

## Step 8: Federal Reserve Examiner Simulation

In [ ]:
print("🏛 FEDERAL RESERVE EXAMINER SIMULATION")
print("=" * 70)

examiner_query = f"Show me the fraud routing decision for FedNow payment {payment_id} including latency compliance"
print(f"**Analysis:** EXAMINER QUERY: {examiner_query}")

examiner_response = backend.format_examiner_response(
    stored_decision_id,
    examiner_query,
    db_backend
)
print(examiner_response)

## Step 9: RTP Compliance Validation

In [ ]:
print("[URGENT] RTP COMPLIANCE VALIDATION")
print("=" * 70)

# Check irrevocability tracking
is_irrevocable = retrieved_decision.tags.get("irrevocable", False)
routing_decision = None
for output in retrieved_decision.outputs:
    if output.name == "routing_decision":
        routing_decision = output.value

if routing_decision == "approve" and is_irrevocable:
    print("[SECURED] [SUCCESS] Irrevocable approval properly flagged")
    print("  • Payment cannot be recalled once processed")
    print("  • Audit trail preserves decision rationale permanently")
elif routing_decision == "reject":
    print("🚫 [SUCCESS] Payment rejected - no irrevocability risk")
else:
    print("[FAILED] Irrevocability flag inconsistent with decision")

# Check latency compliance
actual_latency = retrieved_decision.tags.get("actual_latency_ms")
max_latency = retrieved_decision.tags.get("max_decision_latency_ms", 500)

if actual_latency and int(actual_latency) <= max_latency:
    print(f"\n[URGENT] [SUCCESS] Decision latency compliant: {actual_latency}ms ≤ {max_latency}ms")
else:
    print(f"\n[URGENT] [FAILED] Decision latency non-compliant: {actual_latency}ms > {max_latency}ms")

# Payment rail verification
payment_rail = retrieved_decision.tags.get("payment_rail")
if payment_rail in ["fednow", "rtp"]:
    print(f"\n**Cost:** [SUCCESS] Real-time payment rail confirmed: {payment_rail.upper()}")
    print(f"  • Irrevocable payment system")
    print(f"  • Sub-500ms decision requirement")
    print(f"  • Complete audit trail required")

# Overall RTP compliance validation
required_fields = [
    "regulation",
    "payment_rail",
    "irrevocable",
    "latency_requirement_met",
    "actual_latency_ms"
]

validation_result = backend.validate_regulatory_completeness(
    retrieved_decision,
    required_fields
)

status_icon = "[SUCCESS]" if validation_result['is_compliant'] else "[FAILED]"
status_text = "COMPLIANT" if validation_result['is_compliant'] else "NON-COMPLIANT"

print(f"\n{status_icon} RTP Compliance Status: {status_text}")
print(f"**Results:** Completeness Score: {validation_result['completeness_score']:.1%}")

if validation_result['missing_fields']:
    print(f"[FAILED] Missing Required Fields: {', '.join(validation_result['missing_fields'])}")

## Step 10: Performance and Risk Summary

In [ ]:
print("**Results:** RTP PERFORMANCE & RISK SUMMARY")
print("=" * 50)

print(f"[URGENT] Performance Metrics:")
print(f"  • Decision Latency: {routing_result['decision_latency_ms']}ms")
print(f"  • Compliance Target: ≤500ms")
print(f"  • Performance Status: {'[SUCCESS] COMPLIANT' if routing_result['latency_compliant'] else '[FAILED] NON-COMPLIANT'}")

print(f"\n[SECURED] Irrevocability Status:")
print(f"  • Routing Decision: {routing_result['routing_decision'].upper()}")
print(f"  • Irrevocable: {'YES' if routing_result['irrevocable'] else 'NO'}")
print(f"  • Risk Score: {routing_result['risk_score']}")
print(f"  • Confidence: {routing_result['confidence']}")

print(f"\n**Details:** Audit Trail Status:")
print(f"  • Decision ID: {stored_decision_id}")
print(f"  • Payment ID: {payment_id}")
print(f"  • Timestamp: {payment_data['payment_timestamp']}")
print(f"  • Model Version: {routing_result['model_version']}")

if routing_result['irrevocable']:
    print(f"\n[SECURED] CRITICAL IRREVOCABILITY NOTICE:")
    print(f"  • This payment is FINAL once processed")
    print(f"  • Cannot be recalled or reversed")
    print(f"  • Fraud routing decision permanently captured")
    print(f"  • Complete audit trail available for examination")

print(f"\n[SUCCESS] Real-time payments fraud routing demonstration completed")
print(f"🆔 Decision ID: {stored_decision_id}")
print(f"**Cost:** Payment ID: {payment_id}")

## Summary

### What We Accomplished
[SUCCESS] **Created ultra-fast RTP fraud routing** with complete regulatory compliance

[SUCCESS] **Met critical RTP requirements:**
- Sub-500ms decision latency
- Complete irrevocability tracking
- Immutable audit trail capture
- Federal Reserve guidance compliance

[SUCCESS] **Demonstrated regulatory readiness:**
- Fed examiner query simulation
- Latency compliance monitoring
- Irrevocability risk documentation
- Complete decision reconstruction

### Key RTP Compliance Benefits
- **Speed Compliance**: Automated latency monitoring (<500ms)
- **Irrevocability Tracking**: Clear documentation of final decisions
- **Risk Documentation**: Complete fraud routing rationale
- **Audit Readiness**: Full decision history for Fed examination

### Critical RTP Requirements Met
[URGENT] **Speed Requirements:**
- Sub-500ms decision latency
- Real-time fraud scoring
- Immediate routing decisions
- Latency compliance monitoring

[SECURED] **Irrevocability Management:**
- Clear approval/rejection decisions
- Irrevocable flag tracking
- Risk threshold documentation
- Final decision preservation

**Results:** **Regulatory Documentation:**
- Complete decision context
- Model version tracking
- Performance metrics
- Examiner-ready audit trails

### RTP vs Traditional Payments
| Aspect | Traditional ACH | Real-Time Payments |
|--------|----------------|-------------------|
| **Decision Time** | Minutes/Hours | <500ms |
| **Reversibility** | Reversible | Irrevocable |
| **Fraud Review** | Manual possible | Must be automated |
| **Audit Requirements** | Standard | Enhanced |

### Next Steps in Production
1. **Model Optimization**: Ensure <500ms latency under load
2. **Monitoring**: Real-time latency and accuracy monitoring
3. **Alerting**: Immediate alerts for latency violations
4. **Reporting**: Regular Fed compliance reporting

**Performance**: `{routing_result['decision_latency_ms']}ms` (Target: ≤500ms)  
**Status**: `{'COMPLIANT' if routing_result['latency_compliant'] else 'NON-COMPLIANT'}`  
**Payment**: `{payment_id}`  
**Decision**: `{stored_decision_id}`